### Movie Recommendation System

Content-based movie recommendation system using TF-IDF style text processing, vectorization, and cosine similarity.

In [ ]:
import numpy as np
import pandas as pd

### Loading Dataset

Loading movie metadata and credits datasets.

In [ ]:
credits= pd.read_csv("/kaggle/input/tmdb-movie-metadata/tmdb_5000_credits.csv")

In [ ]:
movies= pd.read_csv("/kaggle/input/tmdb-movie-metadata/tmdb_5000_movies.csv")

In [ ]:
movies.head(1)

In [ ]:
credits.head(1)

In [ ]:
movies.head(1)

In [ ]:
movies=movies.merge(credits, on= "title")

In [ ]:
movies.shape

### Feature Selection

Keeping only important columns required for recommendations.

In [ ]:
movies= movies[["title", "genres", "id", "overview", "keywords", "cast", "crew"]]

In [ ]:
movies.head(1)

In [ ]:
movies.dropna(inplace=True)

In [ ]:
movies.isnull().sum()

In [ ]:
movies.duplicated().sum()

In [ ]:
import ast

In [ ]:
def convert(obj):
    l=[]
    for i in ast.literal_eval(obj):
        l.append(i['name'])
    return l

In [ ]:
movies['genres']= movies['genres'].apply(convert)

In [ ]:
movies['keywords']= movies['keywords'].apply(convert)

In [ ]:
def convert_cast(obj):
    l=[]
    counter=0
    for i in ast.literal_eval(obj):
        if counter!=3:
            l.append(i['name'])
            counter+=1
        else:
            break;    
    return l

In [ ]:
movies['cast']= movies['cast'].apply(convert_cast)

In [ ]:
def fetch_dir(obj):
    l=[]
    for i in ast.literal_eval(obj):
        if i['job']== 'Director':
            l.append(i['name'])
            break
    return l    

In [ ]:
movies['crew']= movies['crew'].apply(fetch_dir)

In [ ]:
movies.head(1)

In [ ]:
movies["overview"]= movies["overview"].apply(lambda x: x.split())

In [ ]:
movies.head(1)

### Data Preprocessing

Cleaning and converting text-based columns into usable format.

In [ ]:
movies["genres"]= movies["genres"].apply(lambda x: [i.replace(" ", "") for i in x])
movies["overview"]= movies["overview"].apply(lambda x: [i.replace(" ", "") for i in x])
movies["keywords"]= movies["keywords"].apply(lambda x: [i.replace(" ", "") for i in x])
movies["cast"]= movies["cast"].apply(lambda x: [i.replace(" ", "") for i in x])
movies["crew"]= movies["crew"].apply(lambda x: [i.replace(" ", "") for i in x])

In [ ]:
movies.head(1)

In [ ]:
movies["tags"]= movies["genres"]+ movies["overview"]+ movies["keywords"]+ movies["cast"]+ movies["crew"]

In [ ]:
df= movies[["id", "title", "tags"]]

In [ ]:
df.head(1)

In [ ]:
df["tags"]= df["tags"].apply(lambda x: " ".join(x))

In [ ]:
df["tags"]= df["tags"].apply(lambda x: x.lower())

In [ ]:
df.head(1)

In [ ]:
!pip install scikit-learn


In [ ]:
from sklearn.feature_extraction.text import CountVectorizer
cv= CountVectorizer(max_features= 5000, stop_words='english')
vector= cv.fit_transform(df['tags']).toarray()

In [ ]:
!pip install nltk

In [ ]:
import nltk 

In [ ]:
df.head(1)

### Text Normalization

Applying stemming to reduce similar words to root form.

In [ ]:
from nltk.stem.porter import PorterStemmer

In [ ]:
ps= PorterStemmer()


In [ ]:
def convert_stem(col):
    L= []
    for i in col.split():
        each= ps.stem(i)
        L.append(each)
    return " ".join(L) 

df['tags']= df['tags'].apply(convert_stem)


In [ ]:
df.head(1)

In [ ]:
from sklearn.metrics.pairwise import cosine_similarity

### Similarity Calculation

Using cosine similarity to find similar movies.

In [ ]:
similarity= cosine_similarity(vector)

In [ ]:
distances_all= sorted(list(enumerate(similarity[0])), reverse=True, key= lambda x: x[1])
fetch_top5= distances_all[1:6]
def fetch_top_func(x):
    for x in fetch_top5:
    print(x[0])

In [ ]:
df['title'][0] 

### Recommendation Function

Function to recommend similar movies based on content similarity.

In [ ]:
def recommend_movie(movie):
    movie_index=  df[df['title']== movie].index[0]
    distances_movie= sorted(list(enumerate(similarity[movie_index])), reverse=True, key= lambda x: x[1])
    fetch_top5= distances_all[1:6]
    for x in fetch_top5:
        print(df['title'][(x[0])])

### Testing the Recommendation System

Testing recommendations using a sample movie title.

In [ ]:
recommend_movie('Avatar')

In [ ]:
df[df['title']== 'Avatar'].index[0] 

### Conclusion

Built a content-based movie recommendation system using vectorization and cosine similarity to recommend similar movies.